# Weighted age-group and sex hypergraph similarity

This analysis calculates weighted `NMIcross` matrices between entry-age groups and between sexes for the chronic-multimorbid cohort using 34 chronic groups and 131 ICD blocks, with and without the two-year observation rule. Repeated patient profiles are represented by patient counts.

In [9]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import mi_hypergraph_functions as mi

TABLES_DIR = Path("tables")
FIGURES_DIR = Path("figures")
TABLES_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

patient_table = pd.read_csv("patient_table.csv")
chronic_memberships = pd.read_csv("chronic_memberships.csv")
block_memberships = pd.read_csv("block_memberships.csv")
chronic_mapping = pd.read_csv("chronic_mapping.csv")
block_mapping = pd.read_csv("block_mapping.csv")

## Define the four analysis conditions

The cohort contains patients with at least two chronic groups (`>= 2`). The same patients are represented using either 34 chronic groups or 131 ICD blocks, and the year restriction is applied independently.

In [10]:
chronic_node_to_id = {
    row.chronic_group: int(row.chronic_group_id) - 1
    for row in (
        chronic_mapping[["chronic_group_id", "chronic_group"]]
        .drop_duplicates()
        .itertuples(index=False)
    )
}
block_node_to_id = {
    row.block_name: int(row.block_id) - 1
    for row in (
        block_mapping[["block_id", "block_name"]]
        .drop_duplicates()
        .itertuples(index=False)
    )
}

representations = {
    "ChronicGroups": {
        "label": "34 chronic groups",
        "memberships": chronic_memberships,
        "node_to_id": chronic_node_to_id,
    },
    "ICDBlocks": {
        "label": "131 ICD blocks",
        "memberships": block_memberships,
        "node_to_id": block_node_to_id,
    },
}

conditions = [
    ("ChronicGroupsChronicCohortAllYears", "ChronicGroups", False),
    ("ChronicGroupsChronicCohort2PlusYears", "ChronicGroups", True),
    ("ICDBlocksChronicCohortAllYears", "ICDBlocks", False),
    ("ICDBlocksChronicCohort2PlusYears", "ICDBlocks", True),
]

age_group_order = sorted(
    patient_table["age_group_10y"].dropna().unique(),
    key=lambda label: int(str(label).split("-")[0].replace("+", "")),
)
sex_order = sorted(patient_table["sex"].dropna().unique())

groupings = {
    "AgeGroups": ("age_group_10y", age_group_order, "Entry-age group"),
    "Sex": ("sex", sex_order, "Sex"),
}

## Construct weighted hypergraphs and calculate NMIcross

Each distinct disease profile is stored once with its patient count. The identity partition keeps every disease node separate, and `NMIcross` permits cross-order projections.

In [11]:
def build_patient_profiles(representation_key, require_two_years):
    representation = representations[representation_key]
    selected = patient_table.loc[
        patient_table["n_chronic_groups"].ge(2),
        ["patient_no", "age_group_10y", "sex", "n_unique_years"],
    ].copy()
    if require_two_years:
        selected = selected.loc[selected["n_unique_years"].ge(2)]

    node_to_id = representation["node_to_id"]
    source_memberships = representation["memberships"]
    memberships = (
        source_memberships.loc[
            source_memberships["patient_no"].isin(selected["patient_no"]),
            ["patient_no", "node_label"],
        ]
        .drop_duplicates(["patient_no", "node_label"])
    )
    profiles = (
        memberships.groupby("patient_no")["node_label"]
        .agg(lambda nodes: tuple(sorted(node_to_id[node] for node in nodes)))
        .rename("profile")
    )
    patients = selected.merge(profiles, on="patient_no", how="inner")
    partition = list(range(len(node_to_id)))  # B = N: 34 or 131
    return patients, partition


def build_hypergraphs(patients, group_column, group_order):
    return {
        group: Counter(
            patients.loc[
                patients[group_column].eq(group), "profile"
            ]
        )
        for group in group_order
    }


def pairwise_nmicross(hypergraphs, partition, group_order):
    matrix = pd.DataFrame(index=group_order, columns=group_order, dtype=float)
    for left_position, left_group in enumerate(group_order):
        for right_group in group_order[left_position:]:
            score = float(
                mi.NMIcross(
                    hypergraphs[left_group],
                    hypergraphs[right_group],
                    partition=partition,
                )
            )
            matrix.loc[left_group, right_group] = score
            matrix.loc[right_group, left_group] = score
    return matrix

In [12]:

def save_heatmap(matrix, output_file, axis_label):
    figure_width = max(5, 0.8 * len(matrix.columns) + 2)
    figure, axis = plt.subplots(figsize=(figure_width, figure_width))
    image = axis.imshow(matrix.to_numpy(), cmap="Oranges", vmin=0, vmax=1)
    axis.set_xticks(
        range(len(matrix.columns)), labels=matrix.columns, rotation=45, ha="right"
    )
    axis.set_yticks(range(len(matrix.index)), labels=matrix.index)
    axis.set_xlabel(axis_label)
    axis.set_ylabel(axis_label)
    for row in range(len(matrix.index)):
        for column in range(len(matrix.columns)):
            value = matrix.iat[row, column]
            axis.text(
                column, row, f"{value:.3f}",
                ha="center", va="center",
                color="white" if value > 0.5 else "black",
            )
    figure.colorbar(image, ax=axis, label="Weighted NMIcross")
    figure.tight_layout()
    figure.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(figure)

## Run age-group matrices and summarize the sex comparisons

Each age-group result is saved as a complete symmetric CSV and heatmap. The Female–Male NMIcross values are saved together in one CSV with one row per representation and observation rule; no sex heatmaps are produced.

In [13]:
nmi_results = {}
sex_comparison_rows = []
for condition_key, representation_key, require_two_years in conditions:
    patients, partition = build_patient_profiles(
        representation_key, require_two_years
    )
    representation_label = representations[representation_key]["label"]
    cohort_label = "at least 2 chronic groups"
    years_label = "at least 2 observed years" if require_two_years else "all observation lengths"

    for grouping_key, (group_column, group_order, axis_label) in groupings.items():
        hypergraphs = build_hypergraphs(patients, group_column, group_order)
        matrix = pairwise_nmicross(hypergraphs, partition, group_order)
        result_key = f"{grouping_key}{condition_key}"
        nmi_results[result_key] = matrix

        if grouping_key == "Sex":
            sex_comparison_rows.append({
                "condition": condition_key,
                "cohort": cohort_label,
                "representation": representation_label,
                "observation_rule": years_label,
                "comparison": f"{group_order[0]} vs {group_order[1]}",
                "nmi_cross": round(float(matrix.iloc[0, 1]), 3),
            })
            continue

        matrix.round(3).to_csv(
            TABLES_DIR / f"nmiCross{result_key}.csv",
            index_label=group_column,
        )
        save_heatmap(
            matrix,
            FIGURES_DIR / f"nmiCross{result_key}.png",
            axis_label,
        )


In [14]:
sex_nmicross_by_cohort = pd.DataFrame(sex_comparison_rows)
sex_nmicross_by_cohort.to_csv(
    TABLES_DIR / "nmiCrossSexByCohort.csv",
    index=False,
)